# Step 5 — Run the ShopSmart Advisor UI from Colab

Starts the Streamlit app (`app.py`) and gives you a **public link** to open it in your browser.

| Mode | What it shows | Needs | Start-up |
|---|---|---|---|
| **Saved** (`MODE = "saved"`) | the team's real saved results for the 20 test questions (Dense search, GPT answers, "Why not?") | nothing extra - CPU is fine | seconds |
| **Full** (`MODE = "full"`) | any question, live | `chunks.parquet` (+ T4 GPU) | 2–5 min on first page load |

Run the cells from top to bottom. The last cell prints the link.


## 1. Settings


In [1]:
MODE = "saved"         # "saved" = 20 test questions from results/, "full" = 4.45M-chunk knowledge base
UPLOAD_UI_FILES = True # True until app.py / src/ui are on GitHub: upload ui_upload.zip when asked


## 2. Code


In [2]:
import os, shutil, zipfile
from google.colab import drive
drive.mount('/content/drive')
OUT_DIR = "/content/drive/MyDrive/why_not_work"   # where title_vectors.npz was saved by Step 4
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists("/content/repo"):
    !git clone -q https://github.com/Nariassa/ShopSmart-Advisor---An-Evidence-Based-Online-Shopping-Agent-AI6127-GP-.git /content/repo
else:
    !cd /content/repo && git pull -q
%cd /content/repo

if UPLOAD_UI_FILES:
    from google.colab import files
    print("Upload ui_upload.zip now")
    for name, data in files.upload().items():
        open("/content/ui_upload.zip", "wb").write(data)
    with zipfile.ZipFile("/content/ui_upload.zip") as zf:
        for member in zf.namelist():
            if member.endswith("/"):
                continue
            target = os.path.join("/content/repo", member.split("/", 1)[1])   # drop the top folder name
            os.makedirs(os.path.dirname(target), exist_ok=True)
            with zf.open(member) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst)
!ls app.py src/ui src/why_not


Mounted at /content/drive
/content/repo
Upload ui_upload.zip now


Saving ui_upload.zip to ui_upload.zip
app.py

src/ui:
backend.py  __init__.py

src/why_not:
__init__.py  why_not.py


In [3]:
!pip install -q streamlit chromadb sentence-transformers transformers pyarrow openai
# A red "pip's dependency resolver" warning is harmless - ignore it.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.


## 3. Data (full mode only)
Skipped automatically in saved mode.


In [4]:
import glob, json

def find_in_drive(name):
    for root in ["/content/drive/MyDrive", "/content/drive/Shareddrives"]:
        for depth in ["", "*/", "*/*/", "*/*/*/"]:
            hits = glob.glob(f"{root}/{depth}{name}")
            if hits:
                return hits[0]
    return None

env = {"SHOPSMART_MODE": MODE}
if MODE == "full":
    CHUNKS = "/content/chunks.parquet"
    if not os.path.exists(CHUNKS):
        src = find_in_drive("chunks.parquet")
        if src:
            print("Found in Drive:", src); shutil.copy(src, CHUNKS)
        else:
            manifest = json.load(open("download_manifest.json"))
            file_id = next(f["id"] for f in manifest["files"] if f["name"] == "chunks.parquet")
            !gdown --fuzzy "https://drive.google.com/uc?id={file_id}" -O {CHUNKS}
    env["CHUNKS_PATH"] = CHUNKS
    env["TITLE_CACHE"] = f"{OUT_DIR}/title_vectors.npz"   # built by Step 4; rebuilt if missing
    env["CHROMA_PATH"] = "/content/chroma_db"
    if not os.path.isdir("/content/chroma_db"):
        z = find_in_drive("chroma_db.zip")
        if z:
            with zipfile.ZipFile(z) as zf: zf.extractall("/content")
    print("chunks:", CHUNKS, "| title index:", os.path.exists(env["TITLE_CACHE"]),
          "| chroma:", os.path.isdir("/content/chroma_db"))
print(env)


{'SHOPSMART_MODE': 'saved'}


## 4. Optional: OpenAI key for generated answers
Press **Enter** without typing to skip; the app then shows evidence quotes instead.


In [5]:
from getpass import getpass
key = getpass("OpenAI API key (Enter to skip): ").strip()
if key:
    env["OPENAI_API_KEY"] = key
print("Answer mode:", "LLM" if key else "evidence quotes")


OpenAI API key (Enter to skip): ··········
Answer mode: evidence quotes


## 5. Start the app and get the link
Open the printed `https://....trycloudflare.com` link. In full mode the **first** page load takes a few minutes (loading 4.45M chunks); after that it is fast.


In [6]:
import subprocess, time, re
os.environ.update(env)
!pkill -f "streamlit run" ; pkill -f cloudflared ; sleep 1
if not os.path.exists("/content/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared

app = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
                       stdout=open("/content/streamlit.log", "w"), stderr=subprocess.STDOUT, env=os.environ.copy())
tunnel = subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8501"],
                          stdout=open("/content/tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
    if m:
        url = m.group(0); break
print("Open the app here:", url or "(no link yet - run this cell again)")


^C
Open the app here: https://floating-operation-lid-keyboard.trycloudflare.com


## 6. If something goes wrong
Shows the last lines of the app log (errors appear here).


In [7]:
!tail -n 30 /content/streamlit.log




2026-11-01 16:10:21.067 Uvicorn server started on :::8501
